1. Tokenization - 1st step of Preprocessing in pretraining the LLM 

2. Preprocessing 
Row text ->
1. split into words/sub-words=token ->
2. convert words into token IDs ->
3. convert token IDs into Vector Representation ->
4. vector embedding 

In [426]:
with open("the-verdict-book.txt", "r", encoding="utf-8") as f:
  row_text = f.read()
no_of_chracter = len(row_text)
print(no_of_chracter)
print(row_text[:50])

20479
I HAD always thought Jack Gisburn rather a cheap g


1. split row text into words

In [427]:
import re 
txt="hello , world? ,guys."
print(txt)
print(re.split(r'(\s)',txt))
# split only by the whitespace


hello , world? ,guys.
['hello', ' ', ',', ' ', 'world?', ' ', ',guys.']


In [428]:
result = re.split(r'([,.]|\s)',txt)
print(result)
# split by , and .and whitespace also

['hello', ' ', '', ',', '', ' ', 'world?', ' ', '', ',', 'guys', '.', '']


In [429]:
# remove the whitspace
result = [items for items in result if items.strip()]
print(result)

['hello', ',', 'world?', ',', 'guys', '.']


In [430]:
print("row_text=",len(row_text))
preprocessed_1=re.split(r'[,.:;?_!"()\'{}]|--|/s',row_text)
print("book characters inside=",len(preprocessed_1),preprocessed_1[:3])
preprocessed_1=[item.strip() for item in preprocessed_1 if item.strip()]
print("after remove free space=",len(preprocessed_1),preprocessed_1[:3])

## it give the sentence between 2 splits as one character 


row_text= 20479
book characters inside= 903 ['I HAD always thought Jack Gisburn rather a cheap genius', 'though a good fellow enough', 'so it was no great surprise to me to hear that']
after remove free space= 748 ['I HAD always thought Jack Gisburn rather a cheap genius', 'though a good fellow enough', 'so it was no great surprise to me to hear that']


In [431]:
print(row_text[:15])
print("row_text=",len(row_text))
preprocessed=re.split(r'[,.:;?!"()\'{}]|--|\s',row_text)
print("book characters inside=",len(preprocessed),preprocessed[:22])
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print("after remove free space=",len(preprocessed),preprocessed[:22])
# it give word


I HAD always th
row_text= 20479
book characters inside= 4582 ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', 'though', 'a', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to']
after remove free space= 3788 ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', 'though', 'a', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to']


## Step 2: Creating Token IDs

1. make the unique token bucket

In [432]:
token = sorted(set(preprocessed))
print(token[:10])
vocab_size=len(token)
print("length = ",vocab_size)
print("last token =",token[-1])
# this much is the unique word inside the token

['A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin']
length =  1128
last token = yourself


2. convert token into token IDs
using dict key and values

In [433]:
vocab= { val:i for i,val in enumerate(token) }

for i,item in enumerate(vocab.items()):
    if i < 10:
     print(item)


('A', 0)
('Ah', 1)
('Among', 2)
('And', 3)
('Are', 4)
('Arrt', 5)
('As', 6)
('At', 7)
('Be', 8)
('Begin', 9)


## step-3 make class of the encode and decode


In [434]:
class simpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab # ('A' ,1)
        self.int_to_str={i:item for i , item in enumerate(vocab)} # (1,'A')
    
    def encode(self,text):
        preprocessed=re.split(r'[,.:;?!"()\'{}]|--|\s',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        ids= [self.str_to_int[s] for s in preprocessed]
        return ids


    def decode(self,interger):
        # input is integers ids 
        token =" ".join([self.int_to_str[i] for i in interger])
        
        return token 

In [435]:
tokenizer = simpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[45, 848, 986, 601, 533, 745, 1124, 595, 56, 27, 849, 1106, 753, 792]


In [436]:
txt=tokenizer.decode(ids)
print(txt)

It s the last he painted you know Mrs Gisburn said with pardonable pride


## Unique Tokenizer introduce


In [437]:
print(row_text[:22])

I HAD always thought J


In [438]:
print(vocab['HAD'])

33


In [439]:
p1=simpleTokenizerV1(vocab)
p1.encode("I HAD always thought") # these words exist in the vocab so it will give the integer ids of these words

[42, 33, 152, 1001]

In [440]:
p1.decode([42,33,152])

'I HAD always'

In [441]:
for i,item in vocab.items():
     if item < 5:
      print(i,item)

A 0
Ah 1
Among 2
And 3
Are 4


In [442]:
vocab['<|unk|>'] = len(vocab)
vocab['<|endoftext|>'] = len(vocab)

In [443]:
for i,item in vocab.items():
    if item >= len(vocab)-5:
     print(i,item)

younger 1125
your 1126
yourself 1127
<|unk|> 1128
<|endoftext|> 1129


there is the <unk> and <endoftext> is the token but it do not handle the unknown words and the end of the sources.

so have to add it inside the class simpleTokenizerV1 

In [444]:
# check the unknown token and end of text token

#1. we explicitly mention the <unk> 
print(p1.encode("I HAD always thought <|unk|> <|endoftext|>"))

# 2. giving the unknown words
s1=p1.encode("hello world mining engineers")
print(s1)

[42, 33, 152, 1001, 1128, 1129]


KeyError: 'hello'

In [445]:
class simpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab # ('A' ,1)
        self.int_to_str={i:item for i , item in enumerate(vocab)} # (1,'A')
    
    def encode(self,text):
        preprocessed=re.split(r'[,.:;?!"()\'{}]|--|\s',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] if s in self.str_to_int else self.str_to_int['<|unk|>'] for s in preprocessed]
        return ids


    def decode(self,interger):
        # input is integers ids 
        token =" ".join([self.int_to_str[i] if i in self.int_to_str else self.int_to_str[self.str_to_int['<|unk|>']] for i in interger])
        
        return token

In [446]:
p2=simpleTokenizerV2(vocab)
s1=p2.encode("hello world mining engineers")
print(s1)
# <|unk|> = 1128

[1128, 1128, 1128, 1128]


In [447]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [448]:
p2.encode(text)

[1128, 357, 1124, 627, 973, 1129, 44, 986, 954, 982, 721, 986, 1128]